# Qwen-VL 检测服务 · 远程测试

本 Notebook 用于对已部署的远程服务进行功能验证。

**当前服务地址：** `http://118.31.37.161:8000`

---

In [ ]:
# ── 依赖 ────────────────────────────────────────────────────────────────────
import base64
import json
from pathlib import Path

import httpx
from IPython.display import Image as IPImage, display, Markdown
from PIL import Image as PILImage
import io

# ── 配置 ────────────────────────────────────────────────────────────────────
SERVICE_URL = "http://118.31.37.161:8000"
SERVICE_KEY = None          # 当前服务未启用鉴权，留 None 即可
TIMEOUT     = 120           # 秒，大模型推理较慢，建议 >= 60

HEADERS = {"X-API-Key": SERVICE_KEY} if SERVICE_KEY else {}

# ── 工具函数 ─────────────────────────────────────────────────────────────────
def img_to_b64(path: str | Path) -> str:
    """将本地图片编码为 Base64 字符串。"""
    return base64.b64encode(Path(path).read_bytes()).decode()

def b64_to_pil(b64: str) -> PILImage.Image:
    """将 Base64 字符串解码为 PIL Image。"""
    return PILImage.open(io.BytesIO(base64.b64decode(b64)))

def show_image(b64: str, caption: str = "", width: int = 700) -> None:
    """在 Notebook 中内联显示 Base64 图片。"""
    if caption:
        display(Markdown(f"**{caption}**"))
    display(IPImage(data=base64.b64decode(b64), width=width))

def detect(image_path: str | Path, prompt: str | None = None) -> dict:
    """调用 /v1/detect 并返回响应 JSON。"""
    payload = {"image_base64": img_to_b64(image_path)}
    if prompt:
        payload["prompt"] = prompt
    r = httpx.post(f"{SERVICE_URL}/v1/detect", json=payload,
                   headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()

print("✅ 初始化完成")

---
## 1. 健康检查

In [ ]:
r = httpx.get(f"{SERVICE_URL}/health", timeout=10)
print(f"HTTP {r.status_code}:", r.json())
assert r.status_code == 200 and r.json()["status"] == "ok", "❌ 健康检查失败"
print("✅ 服务在线")

---
## 2. 图像传输验证（echo-image）

验证图片能正确上传并解析，不调用大模型。

In [ ]:
image_path = "images/vehicle_test.jpg"

r = httpx.post(
    f"{SERVICE_URL}/v1/debug/echo-image",
    json={"image_base64": img_to_b64(image_path)},
    headers=HEADERS,
    timeout=30,
)
r.raise_for_status()
data = r.json()
print(f"图像尺寸：{data['image_width']} × {data['image_height']} 像素")

show_image(data["image_base64"], caption="原始上传图像（echo 返回）")

---
## 3. 目标检测 — 车辆图像（默认 prompt）

In [ ]:
result = detect("images/vehicle_test.jpg")
print(f"检测类型：{result['type']}")

if result["type"] == "detected":
    print(f"检测到 {len(result['objects'])} 个目标：")
    for obj in result["objects"]:
        x1, y1, x2, y2 = obj["bbox_2d"]
        print(f"  [{obj['label']}]  bbox=({x1},{y1})-({x2},{y2})")
    show_image(result["image_base64"], caption="渲染结果（边界框）")
else:
    print("未检测到目标")

---
## 4. 目标检测 — 人物图像（自定义 prompt）

In [ ]:
result = detect("images/person_test.jpg", prompt="检测图中所有的人")
print(f"检测类型：{result['type']}")

if result["type"] == "detected":
    print(f"检测到 {len(result['objects'])} 个目标：")
    for obj in result["objects"]:
        x1, y1, x2, y2 = obj["bbox_2d"]
        print(f"  [{obj['label']}]  bbox=({x1},{y1})-({x2},{y2})")
    show_image(result["image_base64"], caption="渲染结果（人物检测）")
else:
    print("未检测到目标")

---
## 5. 原图与渲染图对比

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

image_path = "images/vehicle_test.jpg"
result = detect(image_path)

original = PILImage.open(image_path)

if result["type"] == "detected":
    rendered = b64_to_pil(result["image_base64"])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(original)
    axes[0].set_title("原图", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(rendered)
    axes[1].set_title(f"检测结果（{len(result['objects'])} 个目标）", fontsize=14)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("未检测到目标，无渲染图")

---
## 6. 错误处理验证

验证服务对异常请求的处理是否符合预期。

In [ ]:
# 6-1: 缺少图像字段 → 应返回 422
r = httpx.post(f"{SERVICE_URL}/v1/detect", json={}, headers=HEADERS, timeout=10)
assert r.status_code == 422, f"期望 422，实际 {r.status_code}"
print(f"✅ 缺少图像字段 → HTTP {r.status_code}（符合预期）")

# 6-2: 无效 Base64 → 应返回 422
r = httpx.post(f"{SERVICE_URL}/v1/detect",
               json={"image_base64": "not_valid_base64!!!"},
               headers=HEADERS, timeout=10)
assert r.status_code == 422, f"期望 422，实际 {r.status_code}"
print(f"✅ 无效 Base64 → HTTP {r.status_code}（符合预期）")

# 6-3: Base64 是合法字符串但不是图像 → 应返回 422
fake_b64 = base64.b64encode(b"this is not an image").decode()
r = httpx.post(f"{SERVICE_URL}/v1/detect",
               json={"image_base64": fake_b64},
               headers=HEADERS, timeout=10)
assert r.status_code == 422, f"期望 422，实际 {r.status_code}"
print(f"✅ 非图像数据 → HTTP {r.status_code}（符合预期）")

print("\n🎉 所有错误处理验证通过")

---
## 7. 使用自定义图像

修改下方 `custom_image_path` 和 `custom_prompt` 即可测试任意图片。

In [ ]:
custom_image_path = "images/vehicle_test.jpg"   # ← 替换为你的图片路径
custom_prompt = None  # ← 留 None 使用默认检测，或填入自定义指令，如 "只检测行人"

result = detect(custom_image_path, prompt=custom_prompt)
print(f"检测类型：{result['type']}")

if result["type"] == "detected":
    print(f"检测到 {len(result['objects'])} 个目标：")
    for obj in result["objects"]:
        print(f"  {obj}")
    show_image(result["image_base64"], caption="渲染结果")
else:
    print("未检测到目标")

---
## 8. 完整测试汇总

In [ ]:
import traceback

results_summary = []

def run_test(name, fn):
    try:
        fn()
        results_summary.append(("✅", name))
    except Exception as e:
        results_summary.append(("❌", f"{name} — {e}"))

run_test("健康检查", lambda: (
    r := httpx.get(f"{SERVICE_URL}/health", timeout=10),
    [None for _ in [None] if r.status_code != 200 and (_ for _ in ()).throw(AssertionError(f"HTTP {r.status_code}"))]
))

# 更简洁的写法
def test_health():
    r = httpx.get(f"{SERVICE_URL}/health", timeout=10)
    assert r.status_code == 200

def test_echo():
    r = httpx.post(f"{SERVICE_URL}/v1/debug/echo-image",
                   json={"image_base64": img_to_b64("images/vehicle_test.jpg")},
                   headers=HEADERS, timeout=30)
    assert r.status_code == 200

def test_detect_vehicle():
    result = detect("images/vehicle_test.jpg")
    assert result["type"] in ("detected", "no_detection")

def test_detect_person():
    result = detect("images/person_test.jpg", prompt="检测图中所有的人")
    assert result["type"] in ("detected", "no_detection")

def test_422_missing():
    r = httpx.post(f"{SERVICE_URL}/v1/detect", json={}, headers=HEADERS, timeout=10)
    assert r.status_code == 422

def test_422_invalid_b64():
    r = httpx.post(f"{SERVICE_URL}/v1/detect",
                   json={"image_base64": "!!invalid!!"}, headers=HEADERS, timeout=10)
    assert r.status_code == 422

results_summary.clear()
for name, fn in [
    ("健康检查",                 test_health),
    ("图像 echo 验证",           test_echo),
    ("车辆检测（默认 prompt）",   test_detect_vehicle),
    ("人物检测（自定义 prompt）", test_detect_person),
    ("422 缺少字段",             test_422_missing),
    ("422 无效 Base64",          test_422_invalid_b64),
]:
    run_test(name, fn)

print("测试结果：\n")
for icon, msg in results_summary:
    print(f"  {icon}  {msg}")

passed = sum(1 for icon, _ in results_summary if icon == "✅")
total  = len(results_summary)
print(f"\n  {passed}/{total} 通过")